In [1]:
# base
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import warnings

# models
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor

# optimization hyperparameters
import optuna
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error

# log
import sys
from pathlib import Path
from src.config.log_files import auto_logger, setup_log

# path
from src.config.path import PROCESS_PATH

/Users/nguyentantai/Desktop/app/ADY201m_Phase02/Drone-Power-predict/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
repo_root = Path.cwd().resolve()
if repo_root.name == "eda":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

logger = setup_log(name="drone_hyperopt", filename="drone_hyperopt")


In [3]:
warnings.filterwarnings('ignore')

df_all = pd.read_parquet(PROCESS_PATH / "flight.parquet", engine="pyarrow")
MISSION_ROUTES = [f"R{i}" for i in range(1, 8)]
df = df_all.loc[df_all["route"].isin(MISSION_ROUTES)].copy().reset_index(drop=True)
print(f"Primary modeling cohort: {len(df)} R1--R7 flights from {len(df_all)} total campaigns")
df["route"].value_counts().sort_index()

Primary modeling cohort: 196 R1--R7 flights from 209 total campaigns


route
R1    182
R2      1
R3      1
R4      1
R5      4
R6      5
R7      2
Name: count, dtype: int64

In [4]:
def log_trial_result(study, trial, value, params):
    logger.info(
        f"Hyperopt trial finished | study={study.study_name} | trial={trial.number} | value={value:.6f} | params={params}"
    )


@auto_logger(logger)
def run_hyperopt(name, obj_fn, X_train, y_train, n_trials=50):
    study = optuna.create_study(
        direction="minimize",
        study_name=f"{name}_drone_energy",
        sampler=optuna.samplers.TPESampler(seed=42),
    )

    logger.info(f"Starting hyperparameter optimization for {name}")
    study.optimize(
        lambda trial: obj_fn(trial, X_train, y_train),
        n_trials=n_trials,
        callbacks=[
            lambda study, trial: log_trial_result(study, trial, trial.value, trial.params)
        ],
    )
    logger.info(
        f"Completed hyperparameter optimization for {name} | best_rmse={study.best_value:.4f} | best_params={study.best_params}"
    )
    return study

In [5]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def preprocess_fold(X, y, train_idx, val_idx):
    scaler = StandardScaler()
    X_train_fold = scaler.fit_transform(X[train_idx])
    X_val_fold = scaler.transform(X[val_idx])
    return X_train_fold, X_val_fold, y[train_idx], y[val_idx]

def objective_ridge(trial, X, y):
    params = {
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
    }
    scores = []
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val, y_tr, y_val = preprocess_fold(X, y, train_idx, val_idx)
        model = Ridge(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        scores.append(root_mean_squared_error(y_val, preds))
    return np.mean(scores)

def objective_ada(trial, X, y):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 1.0, log=True),
        "loss": trial.suggest_categorical("loss", ["linear", "square", "exponential"]),
        "random_state": 42,
    }
    scores = []
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val, y_tr, y_val = preprocess_fold(X, y, train_idx, val_idx)
        model = AdaBoostRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        scores.append(root_mean_squared_error(y_val, preds))
    return np.mean(scores)

def objective_rf(trial, X, y):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "random_state": 42,
        "n_jobs": -1,
    }
    scores = []
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val, y_tr, y_val = preprocess_fold(X, y, train_idx, val_idx)
        model = RandomForestRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        scores.append(root_mean_squared_error(y_val, preds))
    return np.mean(scores)

def objective_xgb(trial, X, y):
    params = {
        "objective": "reg:squarederror",
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "random_state": 42,
        "n_jobs": -1,
        "monotone_constraints": tuple(MONOTONE_CONSTRAINTS),
    }
    scores = []
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val, y_tr, y_val = preprocess_fold(X, y, train_idx, val_idx)
        model = XGBRegressor(**params, eval_metric="rmse")
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        preds = model.predict(X_val)
        scores.append(root_mean_squared_error(y_val, preds))
    return np.mean(scores)

def objective_lgbm(trial, X, y):
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 8, 40),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": 1,
        "min_child_samples": trial.suggest_int("min_child_samples", 3, 30),
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
        "monotone_constraints": MONOTONE_CONSTRAINTS,
    }
    scores = []
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val, y_tr, y_val = preprocess_fold(X, y, train_idx, val_idx)
        model = LGBMRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric="rmse")
        preds = model.predict(X_val)
        scores.append(root_mean_squared_error(y_val, preds))
    return np.mean(scores)

In [6]:
feature_cols = [
    'total_distance_m',      
    'max_height_agl',
    'payload',
    'wind_speed_mean',
    'wind_speed_std',
    'wind_x_mean',
    'wind_y_mean',
    'speed_mean',
    'speed_max',
    'velocity_mag_mean',
    'velocity_mag_max',
    'acceleration_mag_mean',
    'acceleration_mag_std',
] 
target = 'energy_consumed_Wh'

PAYLOAD_IDX = feature_cols.index('payload')
MONOTONE_CONSTRAINTS = [0] * len(feature_cols)
MONOTONE_CONSTRAINTS[PAYLOAD_IDX] = 1

X = df[feature_cols].values
y = df[target].values

X_train_opt, X_test_holdout, y_train_opt, y_test_holdout = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Optimization rows: {len(X_train_opt)} | Untouched holdout rows: {len(X_test_holdout)}")

studies = {}
objectives = {
    "ridge": objective_ridge,
    "ada": objective_ada,
    "rf": objective_rf,
    "xgb": objective_xgb,
    "lgbm": objective_lgbm,
}

for name, obj_fn in objectives.items():
    study = run_hyperopt(name, obj_fn, X_train_opt, y_train_opt, n_trials=50)
    studies[name] = study
    print(f"{name} best RMSE: {study.best_value:.4f} | params: {study.best_params}")

[2026-07-22 01:57:59] [INFO] [1196642857.py:40]: Starting execution of: run_hyperopt


[I 2026-07-22 01:57:59,821] A new study created in memory with name: ridge_drone_energy


[2026-07-22 01:57:59] [INFO] [754449264.py:15]: Starting hyperparameter optimization for ridge


[I 2026-07-22 01:57:59,826] Trial 0 finished with value: 1.375266595128045 and parameters: {'alpha': 0.03148911647956861}. Best is trial 0 with value: 1.375266595128045.


[2026-07-22 01:57:59] [INFO] [754449264.py:2]: Hyperopt trial finished | study=ridge_drone_energy | trial=0 | value=1.375267 | params={'alpha': 0.03148911647956861}


...

[I 2026-07-22 02:02:06,496] Trial 49 finished with value: 1.797906840882753 and parameters: {'num_leaves': 22, 'max_depth': 6, 'learning_rate': 0.04391113815096425, 'feature_fraction': 0.5007250058963768, 'bagging_fraction': 0.5431492214711167, 'min_child_samples': 23, 'n_estimators': 234}. Best is trial 31 with value: 1.2499242030853406.


[2026-07-22 02:02:06] [INFO] [754449264.py:2]: Hyperopt trial finished | study=lgbm_drone_energy | trial=49 | value=1.797907 | params={'num_leaves': 22, 'max_depth': 6, 'learning_rate': 0.04391113815096425, 'feature_fraction': 0.5007250058963768, 'bagging_fraction': 0.5431492214711167, 'min_child_samples': 23, 'n_estimators': 234}


[2026-07-22 02:02:06] [INFO] [754449264.py:23]: Completed hyperparameter optimization for lgbm | best_rmse=1.2499 | best_params={'num_leaves': 29, 'max_depth': 4, 'learning_rate': 0.06660719933011285, 'feature_fraction': 0.6260733587532079, 'bagging_fraction': 0.5852562501375438, 'min_child_samples': 18, 'n_estimators': 434}


[2026-07-22 02:02:06] [INFO] [1196642857.py:40]: Successfully finished: run_hyperopt in 81.2799 seconds


lgbm best RMSE: 1.2499 | params: {'num_leaves': 29, 'max_depth': 4, 'learning_rate': 0.06660719933011285, 'feature_fraction': 0.6260733587532079, 'bagging_fraction': 0.5852562501375438, 'min_child_samples': 18, 'n_estimators': 434}


In [7]:
Ridge(
    alpha=0.45005153282522975,
    random_state=42,
)

AdaBoostRegressor(
    n_estimators=239,
    learning_rate=0.8229272675704623,
    loss='square',
    random_state=42,
)

RandomForestRegressor(
    n_estimators=317,
    max_depth=7,
    min_samples_split=3,
    min_samples_leaf=1,
    max_features='log2',
    n_jobs=-1,
    random_state=42,
)

XGBRegressor(
    max_depth=2,
    learning_rate=0.23940094350114932,
    subsample=0.8005624401197274,
    colsample_bytree=0.6775768375056557,
    reg_alpha=3.7681393049516054e-07,
    reg_lambda=0.00037085351951358946,
    min_child_weight=10,
    n_estimators=331,
    n_jobs=-1,
    random_state=42,
    monotone_constraints=tuple(MONOTONE_CONSTRAINTS),
)

LGBMRegressor(
    num_leaves=29,
    max_depth=4,
    learning_rate=0.06660719933011285,
    feature_fraction=0.6260733587532079,
    bagging_fraction=0.5852562501375438,
    bagging_freq=1,
    min_child_samples=18,
    n_estimators=434,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
    monotone_constraints=MONOTONE_CONSTRAINTS,
)

,num_leaves,29
,max_depth,4
,learning_rate,0.06660719933011285
,n_estimators,434
,min_child_samples,18
,random_state,42
,n_jobs,-1
,feature_fraction,0.6260733587532079
,bagging_fraction,0.5852562501375438
,bagging_freq,1
,verbose,-1


In [8]:
del df
gc.collect()
%whos

Variable                  Type         Data/Info
------------------------------------------------
AdaBoostRegressor         ABCMeta      <class 'sklearn.ensemble.<...>sting.AdaBoostRegressor'>
KFold                     ABCMeta      <class 'sklearn.model_selection._split.KFold'>
LGBMRegressor             type         <class 'lightgbm.sklearn.LGBMRegressor'>
MISSION_ROUTES            list         n=7
MONOTONE_CONSTRAINTS      list         n=13
PAYLOAD_IDX               int          2
PROCESS_PATH              PosixPath    /Users/nguyentantai/Deskt<...>er-predict/data/processed
Path                      type         <class 'pathlib.Path'>
RandomForestRegressor     ABCMeta      <class 'sklearn.ensemble.<...>t.RandomForestRegressor'>
Ridge                     ABCMeta      <class 'sklearn.linear_model._ridge.Ridge'>
StandardScaler            type         <class 'sklearn.preproces<...>ng._data.StandardScaler'>
X                         ndarray      196x13: 2548 elems, type `float64`, 20384 by